In [10]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [11]:
# Create a simple one layer model using a linear layer
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleLinearModel, self).__init__()
        # Define a single linear layer
        self.linear = nn.Linear(input_size, 10)
        self.linear2 = nn.Linear(10, 20)
        self.linear3 = nn.Linear(20, 15)
        self.linear4 = nn.Linear(15, output_size)

    def forward(self, x):
        # Pass input through the linear layer
        output = self.linear(x)
        output = self.linear2(output)
        output = self.linear3(output)
        output = self.linear4(output)
        return output


# Example usage
model = SimpleLinearModel(input_size=10, output_size=5)
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleLinearModel(
  (linear): Linear(in_features=10, out_features=10, bias=True)
  (linear2): Linear(in_features=10, out_features=20, bias=True)
  (linear3): Linear(in_features=20, out_features=15, bias=True)
  (linear4): Linear(in_features=15, out_features=5, bias=True)
)
Model weights:
linear.weight: torch.Size([10, 10])
  Weight values (first 5): tensor([-0.0555, -0.1237,  0.1147, -0.1763,  0.0265], grad_fn=<SliceBackward0>)
linear.bias: torch.Size([10])
  Bias values (first 5): tensor([ 0.2858,  0.1065,  0.2687, -0.2645, -0.2389], grad_fn=<SliceBackward0>)
linear2.weight: torch.Size([20, 10])
  Weight values (first 5): tensor([-0.1318,  0.0613,  0.0937, -0.0017, -0.0069], grad_fn=<SliceBackward0>)
linear2.bias: torch.Size([20])
  Bias values (first 5): tensor([ 0.0791,  0.0499, -0.2989, -0.1347,  0.1364], grad_fn=<SliceBackward0>)
linear3.weight: torch.Size([15, 20])
  Weight values (first 5): tensor([-0.1155, -0.0845, -0.1574,  0.1516, -0.0797], grad_fn=<SliceBackward0>)
linear3.

In [12]:
# export to onnx
onnx.export(model, torch.randn(1, 10), "simple_linear_model.onnx", export_params=True, opset_version=11)

In [ ]:
# run the model with pytorch
input_data = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]])
with torch.no_grad():
    output = model(input_data)
print(output)

tensor([[ 0.0444,  0.6691,  0.1305,  0.0200, -0.5708]])


In [14]:
# Conv model
# Define a simple model with only Conv1D layer
class SimpleConv1DModel(nn.Module):
    def __init__(self):
        super(SimpleConv1DModel, self).__init__()
        # Single Conv1D layer: 1 input channel, 8 output channels, kernel size 3
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=8, kernel_size=3)

    def forward(self, x):
        return self.conv1(x)

In [15]:
# Create model instance
model = SimpleConv1DModel()

# Set model to evaluation mode
model.eval()

# Create dummy input tensor (batch_size=1, channels=1, sequence_length=10)
dummy_input = torch.randn(1, 1, 10)

In [16]:
# Export to ONNX
torch.onnx.export(
    model,  # model being run
    dummy_input,  # model input
    "simple_conv1d_model.onnx",  # where to save the model
    export_params=True,  # store the trained parameter weights
    opset_version=11,  # the ONNX version to export to
    do_constant_folding=True,  # whether to execute constant folding
    input_names=["input"],  # the model's input names
    output_names=["output"],  # the model's output names
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},  # variable length axes
)

In [17]:
input_values = torch.arange(1, 11, dtype=torch.float32)  # Creates [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
dummy_input = input_values.unsqueeze(0).unsqueeze(0)  # Shape: [1, 1, 10]

print("Input tensor:")
print(dummy_input)
print("Input shape:", dummy_input.shape)

# Pass input through the model
with torch.no_grad():
    output = model(dummy_input)

print("\nOutput tensor:")
print(output)
print("Output shape:", output.shape)

Input tensor:
tensor([[[ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.]]])
Input shape: torch.Size([1, 1, 10])

Output tensor:
tensor([[[ -0.6096,  -0.4480,  -0.2863,  -0.1247,   0.0370,   0.1986,   0.3603,
            0.5219],
         [ -1.2457,  -2.0420,  -2.8384,  -3.6347,  -4.4310,  -5.2274,  -6.0237,
           -6.8200],
         [ -1.0744,  -1.1999,  -1.3253,  -1.4507,  -1.5762,  -1.7016,  -1.8270,
           -1.9525],
         [ -2.0286,  -3.2661,  -4.5037,  -5.7413,  -6.9788,  -8.2164,  -9.4540,
          -10.6915],
         [  1.0944,   1.8605,   2.6266,   3.3927,   4.1589,   4.9250,   5.6911,
            6.4573],
         [ -0.9929,  -1.8631,  -2.7334,  -3.6036,  -4.4738,  -5.3441,  -6.2143,
           -7.0846],
         [  1.1878,   1.6785,   2.1692,   2.6600,   3.1507,   3.6414,   4.1322,
            4.6229],
         [  0.9954,   1.6974,   2.3994,   3.1013,   3.8033,   4.5053,   5.2073,
            5.9093]]])
Output shape: torch.Size([1, 8, 8])
